# 🧪 Lab 1.1 & 1.3 : OLAP, Stockage en Colonnes et BigQuery Console UI

Dans ce lab pratique, vous allez directement utiliser l'interface web **BigQuery Console** (Studio) sur Google Cloud Platform (`console.cloud.google.com/bigquery`).

L'objectif est d'observer et de mesurer la différence fondamentale d'une base analytique (**OLAP**) : le **stockage en colonnes** et le modèle de facturation associé (**bytes scanned / octets traités**).

### 🎯 Objectifs :
1. Ouvrir l'éditeur de requêtes SQL dans la console BigQuery sur le projet `lasalle-big-data`.
2. Observer l'indicateur d'audit (**Dry Run automatique**) en bas à gauche  de l'éditeur SQL : *« Cette requête traitera X Go lors de son exécution »*.
3. Mesurer l'impact financier et technique du stockage en colonnes (`SELECT *` vs `SELECT colonnes_spécifiques`).
4. Analyser les détails d'exécution (panneau d'information d'exécution, parallélisme des slots, timeline).
5. Interroger un jeu de données réel volumineux (**NYC Taxi Trips**).

---

## 0. Accès à la Console BigQuery

1. Rendez-vous sur votre navigateur à l'adresse suivante :
   👉 **[https://console.cloud.google.com/bigquery?project=lasalle-big-data](https://console.cloud.google.com/bigquery?project=lasalle-big-data)**
2. Vérifiez en haut à gauche que le projet sélectionné est bien **`lasalle-big-data`**.
3. Cliquez sur le bouton **`+ Saisir une nouvelle requête`** (ou l'onglet éditeur SQL vide).

> 💡 **Astuce sur le Dry Run dans l'UI** :
> Dès que vous collez une requête SQL valide dans l'éditeur, regardez tout en bas à gauche de l'onglet : une **coche verte** s'affiche avec le texte :
> `Cette requête traitera X Go lors de son exécution.`
> **BigQuery calcule cela avant même que vous ne cliquiez sur Exécuter (Run) ! C'est gratuit et instantané.**

---
## 1. Démonstration du Stockage en Colonnes (Columnar Storage)

Dans une base transactionnelle (**OLTP**, PostgreSQL, MySQL), les données sont rangées **ligne par ligne**.
Dans BigQuery (**OLAP**), les données sont stockées **colonne par colonne** (moteur Capacitor).
Si une table fait 50 Go répartis sur 20 colonnes et que vous ne demandez que 2 colonnes, BigQuery **n'ouvre physiquement que ces 2 colonnes** et ne vous facture que quelques Mo/Go !

### Étape 1.1 : La mauvaise pratique — `SELECT *`

Copiez et collez la requête suivante dans l'éditeur BigQuery, **sans cliquer sur Exécuter** :

```sql
-- ⚠️ MAUVAISE PRATIQUE SUR BIGQUERY : SELECT * scanne TOUTES les colonnes du disque
SELECT *
FROM `bigquery-public-data.wikipedia.pageviews_2026`
WHERE datehour >= '2023-01-01'
LIMIT 10;
```

🔍 **Observation attendue :**
- Regardez la coche verte en haut à droite : même avec un `LIMIT 10`, le volume affiché est colossal (plusieurs **Go** ou **dizaines de Go**) !
- ⚠️ **Règle d'or BigQuery** : `LIMIT 10` ne réduit **JAMAIS** le coût d'une requête, car BigQuery doit d'abord scanner l'ensemble des colonnes sur disque avant de filtrer les 10 premières lignes.

### Étape 1.2 : La bonne pratique analytique — Sélection ciblée des colonnes

Remplacez maintenant le code dans l'éditeur par cette requête optimisée :

```sql
-- ✅ BONNE PRATIQUE : Ne lire que les colonnes nécessaires (title, views)
SELECT 
    title,
    SUM(views) AS total_views
FROM `bigquery-public-data.wikipedia.`
WHERE date >= '2023-01-01'
GROUP BY title
ORDER BY total_views DESC
LIMIT 10;
```

🔍 **Observation & Exécution :**
1. Regardez à nouveau la coche verte : observez la chute drastique du volume scanné.
2. Cliquez sur le bouton bleu **`▷ Exécuter` (Run)**.
3. Observez le résultat (le Top 10 des pages Wikipédia les plus consultées).
4. Cliquez sur l'onglet **Informations sur l'exécution** (Execution details) sous la requête pour observer les étapes du calcul distribué.

---
## 2. Analyse sur un Dataset Massif : NYC Yellow Taxi Trips

Interrogeons la table des courses de taxis new-yorkais (`tlc_yellow_trips_2022`).
Nous souhaitons analyser le comportement des passagers : distance moyenne, montant moyen de la course et nombre total de trajets selon le nombre de passagers dans le taxi.

Collez cette requête SQL dans un nouvel onglet de requête :

```sql
-- Analyse de la distribution des courses selon le nombre de passagers
SELECT 
    passenger_count,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
    ROUND(AVG(fare_amount), 2)   AS avg_fare_usd,
    COUNT(1)                     AS total_trips
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022`
WHERE passenger_count IS NOT NULL
GROUP BY passenger_count
ORDER BY total_trips DESC;
```

### 📊 Décryptage des Métriques d'Exécution dans la Console

Une fois la requête exécutée, regardez en bas dans les onglets de résultats :

1. **Onglet « Résultats de la requête » (Query Results)** :
   - Les courses avec 1 seul passager représentent l'écrasante majorité des trajets (~28 millions de trajets).
2. **Onglet « Informations sur l'exécution » (Job Information / Execution details)** :
   - **Octets traités (Bytes billed / Bytes processed)** : Le volume exact lu sur le stockage distribué.
   - **Temps écoulé (Elapsed time)** : Le temps réel (ex: ~1.2 seconde).
   - **Temps processeur total (Slot time consumed)** : Le temps cumulé de tous les CPUs (ex: ~15 secondes).
   - 👉 **Rapport Slot Time / Elapsed Time** = Degré de parallélisation (ex: 15s / 1.2s $\approx$ **12 processeurs ont travaillé en parallèle pour vous !**).

---
## 3. Optionnel : Vérification en Ligne de Commande (Cloud Shell / `bq`)

Si vous ouvrez le **Cloud Shell** (icône `>_` tout en haut à droite de la console GCP), vous pouvez aussi lancer le Dry Run en ligne de commande avec le CLI officiel `bq` :

```bash
bq query \
  --use_legacy_sql=false \
  --dry_run \
  "SELECT passenger_count, AVG(fare_amount) FROM \`bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022\` GROUP BY 1"
```

Le terminal vous répondra instantanément sans facturation :
`Query successfully validated. Assuming the tables are not modified, running this query will process XXXXX bytes of data.`

---
## 4. Exercice Pratique : À vous de jouer ! 🏋️

### Énoncé :
Dans l'éditeur BigQuery sur la table `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022` :

1. Rédigez une requête SQL qui affiche pour chaque type de paiement (`payment_type`) :
   - Le nombre total de courses (`COUNT(1) AS total_trips`)
   - Le pourboire moyen (`ROUND(AVG(tip_amount), 2) AS avg_tip`)
   - Le montant total facturé (`ROUND(SUM(total_amount), 2) AS total_revenue`)
2. Triez par le nombre de courses décroissant (`ORDER BY total_trips DESC`).
3. **Avant de cliquer sur Exécuter** : notez le volume de données estimé par la coche verte.
4. Exécutez la requête et analysez quel mode de paiement génère les pourboires les plus élevés.

### 💡 Solution SQL (à masquer/consulter après essai) :

```sql
SELECT 
    payment_type,
    COUNT(1)                      AS total_trips,
    ROUND(AVG(tip_amount), 2)     AS avg_tip,
    ROUND(SUM(total_amount), 2)   AS total_revenue
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022`
WHERE payment_type IS NOT NULL
GROUP BY payment_type
ORDER BY total_trips DESC;
```

---
## ✅ Checkpoint de Validation du Module

- [ ] J'ai accédé à la console BigQuery sur le projet `lasalle-big-data`.
- [ ] J'ai compris le principe du stockage en colonnes (Capacitor) et pourquoi `SELECT *` est proscrit sur un Data Warehouse analytique.
- [ ] J'ai repéré l'indicateur d'audit (**Dry Run**) dans la console BigQuery qui estime gratuitement le volume avant exécution.
- [ ] J'ai compris que `LIMIT` ne réduit pas les octets scannés sur BigQuery.
- [ ] J'ai exploré l'onglet *Informations sur l'exécution* (Temps réel vs Slot Time / parallélisme).